## **Music Recommendation Algorithm Project**
</br>
Info here

***Model***

Details

---
### **1. Imports**

In [ ]:
# Importing sys to ensure proper environment setup
import sys

print(sys.version_info)

In [ ]:
# Importing pandas and numpy for numerical analysis
# Importing pyplot and seaborn to visualize the data
# Importing os, pathlib, and warnings for functionality, faster loading, flagging exceptions, etc.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker, pylab
from matplotlib.legend import Legend
import statistics
from scipy.stats import skew, kurtosis, trim_mean
import seaborn as sns
import lightgbm as lgb
import os # possibly remove
from pathlib import Path, PureWindowsPath
from scipy.stats import multivariate_normal, norm, trim_mean, zscore
import warnings
warnings.filterwarnings('ignore')

# Importing sklearn items for preprocessing, model training & testing
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error, classification_report, silhouette_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

from kneed import KneeLocator, DataGenerator, find_shape

# Different label assignment (assign_labels="cluster_qr") as deterministic partitioning alternative
from sklearn.cluster import KMeans, AffinityPropagation, DBSCAN, SpectralClustering
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

# Makes graphs appear in line
%matplotlib inline

sns.set(style="whitegrid", palette="Set3", font_scale=1.25)    # alt muted, spectral, tab20c

print("Setup Complete")

---
### **2. Load Data & Quick Review**

In [ ]:
# Explicitly noting path as being in Windows format to avoid issues with backsplash
#filename = PureWindowsPath("..Data\tcc_ceds_music.csv")

# Convert path to the correct format
#file_path = Path(filename)
rec_test = open("C:\\Users\\winni\\music-rec-algo\\Data\\rec_test_data.csv")
clean_d = open("C:\\Users\\winni\\music-rec-algo\\Data\\clean_d.csv")

# Loading data as a DataFrame
df_rec_test = pd.read_csv(rec_test) 
df_clean_d = pd.read_csv(clean_d)

# Using head() function to display the first five rows of the data
print("Heads")
print("Recommendation Test Data:", df_rec_test.head())
print("Cleaned Data:", df_clean_d.head())

In [ ]:
# Using shape() function to return a tuple listing number of rows and columns in the data

print("Shape of Rec Test:", df_rec_test.shape)
print("Shape of Clean Data:", df_clean_d.shape)

In [ ]:
# Using info() function to view column names, data types, and other relevant information

df_rec_test.info()

df_clean_d.info()

In [ ]:
# Updating column names to remove whitespace and erroroneous characters on rec_test

df_rec_test.columns = df_rec_test.columns.str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(':', '')
df_rec_test.columns = df_rec_test.columns.str.replace(')', '').str.replace('-', '_').str.replace('/', '_')
print("Updated Column Names:", df_rec_test.columns)
print(df_rec_test.info())

--- 
### **3. Feature Engineering: Train/Test/Split**

In [ ]:
# Creating copies of datasets for multiple model testings
df_cl = df_clean_d.copy()
df_rec = df_rec_test.copy()

# Implementing apply() method using lambda to extract float values
df_cl = df_cl.apply(lambda p: float(''.join(filter(str.isdigit, p)))if not p.isnumeric() else float(p))
df_rec = df_rec.apply(lambda p: float(''.join(filter(str.isdigit, p)))if not p.isnumeric() else float(p))

# Setting these as X_train & rec
X_train = df_cl.copy()
X_rec = df_rec.copy()

##### **<p style="text-align:center;">3A. The Body Beautiful: Elbow, Silhouette & More</p>**    
Performing primliminary clustering via the Elbow Method and related methologies.

In [ ]:
# Setting these as X_train & rec
X_train = df_cl.copy()
X_rec = df_rec.copy()

In [ ]:
# Kneed sample generator & plotting

x, y = DataGenerator.figure2()

k1 = KneeLocator(x, y, curve="concave", direction="increasing")
print(k1.knee)
print(k1.knee_y)

# Alternative using find_shape to auto-detect curve type & direction
direction, curve = find_shape(x, y)
k1 = KneeLocator(x, y, curve=curve, direction=direction)

# Using kneedle.knee & kneedle.elbow to store maximum curvature points
kneedle = KneeLocator(x, y, S=1.0, curve="concave", direction="increasing")
print(round(kneedle.knee, 3))
print(round(kneedle.elbow, 3))

# Identifying the y value at the knee
print(round(kneedle.knee_y, 3))

# Normalized data, normalized knee, & normalized distance curve
kneedle.plot_knee_normalized()

# Raw data and knee
kneedle.plot_knee()

In [ ]:
# Inertia score will be stored in this list
inertia_score = []

# Defining cluster values to test
# KMeans model testing 2-12 clusters
k_values = range(2, 13)

# Iterating for loop of cluster possibilities
for k in k_values:
    # Creating initial KMeans model with max # of clusters
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=12)

    # Fitting model on training feature matrix
    kmeans.fit(X_train)

    # Beginning intertia score (how compact clusters are)
    inertia_score.append(kmeans.inertia_)

# Using KneeLocator as backup to identify elbow point of the curve programmatically
k1 = KneeLocator(range(2, 13), inertia_score, curve="convex", direction="decreasing")
k1.elbow

plt.figure(figsize=(14,6))

plt.plot(range(2, 13), inertia_score, linewidth=2, marker=8)
plt.title("Elbow Plot [KMeans] Inertia Score")
plt.xlabel("K")
plt.ylabel("Inertia Score")
plt.xticks(range(2, 13))
plt.show()

# saving plot as image in docs folder
# plt.savefig("C:\\Users\\winni\\music-rec-algo\\FOLDER")

In [ ]:
# Silhouette score will be stored in this list
silhouette = []

# Iterating for loop of cluster possibilities
for k in range(2, 13):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=12)

    predictions = kmeans.fit_predict(X.values)
    silhouette.append(metrics.silhouette_score(X, predictions))

plt.figure(figsize=(14, 6))

plt.plot(range(2, 13), silhouette, linewidth=2, marker=8)
plt.title("Silhouette Plot [KMeans]")
plt.xlabel("K")
plt.ylabel("Silhouette Score")
plt.xticks(k_values)
plt.show()

# saving plot as image in docs folder
# plt.savefig("C:\\Users\\winni\\music-rec-algo\\FOLDER")

In [ ]:
# Creating 2D cluster visual Elbow/Silhouette to help finalize decision

labels = kmeans.fit_predict(X.values)
print(labels)

kmeans.cluster_centers_

plt.figure(figsize=(15, 9))

plt.scatter(X.value[:, 0], X.values[:, 1], c=kmeans.labels_, s=106)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], color='red', s=225)
plt.title("Cluster of Songs")
plt.xlabel("Score")
plt.ylabel("Len")
plt.show()

##### *Elbow & Silhouette Insights*

abc

##### **<p style="text-align:center;">3B. Choosing Clusters</p>**    
Performing encoding separately to minimize chance of error.

In [ ]:
# Label 


In [ ]:
# Drop


In [ ]:
# Strong positive


---
### **4. Modeling Music Features**

In [ ]:
# Model test ... apply KMeans to music

# result should be songs

In [ ]:
# Grouping rating columns together to represent music track attributes
rating = ['dating', 'family_gospel', 'communication', 'family_spiritual', 'like_girls', 
          'shake_the_audience', 'movement_places', 'light_visual_perceptions']

X = songs['rating']

# Apply KMeans clustering with k=3 on selected features
kmeans_music = kmeans(n_clusters=3, random_state=42)
kmeans_music.fit(X)

# Box plot of rating columns
plt.figure(figsize=(12, 6))
sns.boxplot(data=df['rating'], orient='h', palette='spectral')
plt.title('Box Plot of Rating Columns')
plt.xlabel('Rating Score')
plt.ylabel('Columns')
plt.show()

In [ ]:
# Using sample of 10 unlabeled songs to see number of cluster generation
sample_songs = songs.sample(10)

selected_predictors = sample_songs[rating]
sample_labels = kmeans_music.predict(selected_predictors)

# sample_songs[pred] = sample_labels

In [ ]:
# Handling outliers with IQR capping at 1.5*IQR boundary (Winsorization)
# Done in lieu of dropping rows, thereby preserving data

for col in skewed_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df1_log_t[col] = df1_log_t[col].clip(lower=lower, upper=upper)

# Using log1p again to handle zero values safely
# Only appling it to positively skewed columns (skew > 0.5)
for col in skewed_cols:
    if skewness[col] > 0.5:
        df1_log_t[col] = np.log1p(df1_log_t[col])
    elif skewness[col] < -0.5:
        # Reflect then log for negatively skewed
        df1_log_t[col] = np.log1p(df1_log_t[col].max() - df1_log_t[col])

In [ ]:
# Checking for any missing values after skew & transformations
print(df1_log_t.isnull().sum())

# For numeric columns, fill with median (robust to outliers)
num_cols = df1_log_t.select_dtypes(include='number').columns
df1_log_t[num_cols] = df1_log_t[num_cols].fillna(df1_log_t[num_cols].median())

# For categorical columns, fill with mode
cat_cols = df1_log_t.select_dtypes(include='object').columns
for col in cat_cols:
    df1_log_t[col] = df1_log_t[col].fillna(df1[col].mode()[0])

In [ ]:
# Scaling columns
cols_to_scale = df1_log_t.select_dtypes(include=['int', 'float']).columns.tolist()
scaler = StandardScaler()

df1_scaled = df1_log_t.copy()
df1_scaled[cols_to_scale] = scaler.fit_transform(df1_log_t[cols_to_scale])

print(df1_scaled.head(5))

---
### **5. Verifying Changes**

In [ ]:
# Confirm no missing values remain
print("Missing values after cleaning:")
print(df1_scaled.isnull().sum())
print()

# Confirm shape is intact
print("DataFrame shape:\n", df1_scaled.shape)

# Numeric columns check
print("\nNumeric column stats:")
print(df1_scaled[num_cols].describe())

---
### **6. Save Cleaned CSV**

In [ ]:
# Explicity noting path as being in Windows format so I can use forward slash
clean = PureWindowsPath("C:\\Users\\Winni\\music-rec-algo\\Data\\cleaned.csv")

# Convert path to the correct format
file_path = Path(clean)

# Saving data as a DataFrame
df1_scaled.to_csv(file_path, index=False)
# index=False prevents pandas from saving row numbers as a column
print("Cleaned data saved successfully!")

In [ ]:
# Most reliable confirmation is to reload saved file and review it
clean = PureWindowsPath("C:\\Users\\Winni\\music-rec-algo\\Data\\cleaned.csv")
file_path = Path(clean)

df1_clean = pd.read_csv(file_path)

print("Shape:", df1_clean.shape)
print("Missing values:\n", df1_clean.isnull().sum())
print("\nFinal columns:", df1_clean.columns.tolist())
print("\nPreview:\n", df1_clean.head(5))